In [ ]:
import torch
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from torch.utils.data import Dataset,DataLoader
import torch.nn as nn


In [ ]:
df= pd.read_csv('fashion-mnist_train.csv')


In [ ]:
## checking for GPU

device  =  torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using {torch.cuda.get_device_name(0)}")

In [ ]:
df

In [ ]:
X_train,X_test,y_train,y_test  = train_test_split(df.iloc[:,1:].values,df.iloc[:,0].values,test_size=0.2,random_state=42)

In [ ]:
X_train = X_train/255.0
X_test = X_test/255.0


In [ ]:
## custom dataset class

class CustomDataset(Dataset):
    def __init__(self,features,labels):    
        self.features = torch.tensor(features,dtype=torch.float32)
        self.labels = torch.tensor(labels,dtype=torch.long)
    def __len__(self):
        return self.features.shape[1]
    def __getitem__(self, index):
        return self.features[index],self.labels[index]



In [ ]:
# creating NN


class MyNN(nn.Module):


    def __init__(self,num_of_feat):
        super().__init__()

        self.model = nn.Sequential(
            nn.Linear(num_of_feat,128),
            nn.BatchNorm1d(128), ## applying batch norm for trainnig stability
            nn.ReLU(),
            nn.Dropout(p=0.3),
            nn.Linear(128,64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64,10),
            # nn.Softmax() ## no need of explicitly define the softamx the bce has already integrated it
        )


    def forward(self,features):

        return self.model(features)







In [ ]:
## creating NN class for optuna hyperparameter tunning

class MYnn(nn.Module):

    def __init__(self,input_dim,output_dim,num_hidden_layers,neurons_per_layer):

        super().__init__()
    ## creating a dyanamic list which gonna store layers 

        layer=[]

        for i in range(num_hidden_layers):
            layer.append(nn.Linear(input_dim,neurons_per_layer))
            layer.append(nn.BatchNorm1d(neurons_per_layer))
            layer.append(nn.ReLU())
            layer.append(nn.Dropout(0.3))
            input_dim = neurons_per_layer

        ## sepratley adding output layer 

        layer.append(nn.Linear(neurons_per_layer,output_dim))
        layer.append(nn.Softmax())

        self.model = nn.Sequential(*layer)

    def forward(self,x):
        return self.model(x)




In [ ]:
import optuna

In [ ]:
## def objective function

def objective(trial):

## hyperparameter values for the seach space 

    num_hidden_layer = trial.suggest_int('num_hidden_layer',1,5)
    neurons_per_layer = trial.suggest_int('neurons_per_layer',8,128,step=8)

## model init 

    input_dim = 784 
    output_dim = 10

    model = MYnn(input_dim,output_dim,num_hidden_layer,neurons_per_layer)
    model.to(device)

#3 params init

#3 setting learning rate and epochs

    learning_rate= 0.01

    epochs = 50


## optimizer selection

    criterion =  nn.CrossEntropyLoss()
    optim = torch.optim.SGD(model.parameters(),lr=learning_rate,weight_decay=1e-4)



## training loop

    for epoch in range(epochs):

        total_epoch_loss= 0

        for batch_features,batch_labels in train_dataloader:

            batch_features,batch_labels=batch_features.to(device),batch_labels.to(device)



            ## forward pass

            output = model(batch_features)

            ##loss

            loss = criterion(output,batch_labels)
            

            ## backward 
            optim.zero_grad()
            loss.backward()

            ## updating the weights

            optim.step()


            total_epoch_loss = total_epoch_loss + loss.item()

            avg_loss = total_epoch_loss/len(train_dataloader)

            print(f"Epoch {epoch + 1} ,loss : {avg_loss}")











In [ ]:
## creating model object

model = MyNN(X_train.shape[1])

model.to(device)

## loss

criterion =  nn.CrossEntropyLoss()


##

optim = torch.optim.SGD(model.parameters(),lr=learning_rate,weight_decay=1e-4)

In [ ]:
## datset object
train_dataset = CustomDataset(X_train,y_train)
test_dataset = CustomDataset(X_test,y_test)

In [ ]:
## dataloader object

train_dataloader = DataLoader(train_dataset,batch_size=128,shuffle=True,pin_memory=True)
test_dataloader = DataLoader(test_dataset,batch_size=128,shuffle=False,pin_memory=True)

In [ ]:

## trainnig loop

for epoch in range(epochs):

    total_epoch_loss= 0

    for batch_features,batch_labels in train_dataloader:

        batch_features,batch_labels=batch_features.to(device),batch_labels.to(device)



        ## forward pass

        output = model(batch_features)

        ##loss

        loss = criterion(output,batch_labels)
        

        ## backward 
        optim.zero_grad()
        loss.backward()

        ## updating the weights

        optim.step()


        total_epoch_loss = total_epoch_loss + loss.item()

        avg_loss = total_epoch_loss/len(train_dataloader)

        print(f"Epoch {epoch + 1} ,loss : {avg_loss}")



        


         




In [ ]:
#3 changing model to eval mode 

model.eval()

In [ ]:
total = 0
correct = 0

for batch_features, batch_labels in test_dataloader:

    output = model(batch_features.to(device))

    _, predicted = torch.max(output, 1)

    total += batch_labels.to(device).shape[0]

    correct += (predicted == batch_labels.to(device)).sum().item()

print(correct / total)